# Task 1 — Source Extraction

This notebook documents the active sources and the code used to rebuild the raw snapshot. The normal project run uses the frozen files already stored under `data/raw/`; re-harvesting is optional because APIs can change over time.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import extract
from src.config import get_paths, load_config

config = load_config(ROOT)
paths = get_paths(config).ensure()


## Active raw sources

- KAUST repository CSV: 2023
- KAUST Crossref API snapshot: 2024–2025
- KFUPM Pure OAI-PMH / MODS: 2023–2026
- KSU annual open-data JSON: 2023–2025

The older KFUPM EPrints thesis files are retained only as historical raw evidence and are not used by the active pipeline.

In [2]:
checks = {
    "KAUST repository": paths.raw / config["sources"]["kaust_repository"]["file"],
    "KFUPM Pure directory": paths.raw / config["sources"]["kfupm_pure"]["directory"],
}

for label, path in checks.items():
    print(f"{label}: {path.exists()} -> {path.relative_to(ROOT)}")

print("KAUST Crossref pages:", len(list(paths.raw.glob(config["sources"]["kaust_crossref"]["pattern"]))))
for year, filename in config["sources"]["ksu"]["files"].items():
    print(f"KSU {year}: {(paths.raw / filename).exists()} -> data/raw/{filename}")


KAUST repository: True -> data\raw\KAUST_2023_raw.csv
KFUPM Pure directory: True -> data\raw\kfupm_pure
KAUST Crossref pages: 2
KSU 2023: True -> data/raw/ksu_publications_2023_original.json
KSU 2024: True -> data/raw/ksu_publications_2024_original.json
KSU 2025: True -> data/raw/ksu_publications_2025_original.json


## Optional re-harvest

Keep `REHARVEST = False` for normal grading/reproducibility. Set it to `True` only when intentionally refreshing public API data.

In [3]:
REHARVEST = False

if REHARVEST:
    crossref_cfg = config["sources"]["kaust_crossref"]
    extract.fetch_crossref(
        ror_id="01q3tbs38",
        from_year=int(crossref_cfg["from_year"]),
        until_year=int(crossref_cfg["until_year"]),
        output_dir=paths.raw,
    )

    pure_dir = paths.raw / config["sources"]["kfupm_pure"]["directory"]
    for year in range(config["years"]["min"], config["years"]["max"] + 1):
        pages = extract.fetch_pure_pages(
            base_url="https://pure.kfupm.edu.sa/ws/oai",
            year=year,
            output_dir=pure_dir,
        )
        print(f"KFUPM Pure {year}: {pages} pages")
else:
    print("Using the frozen raw snapshot; no network calls were made.")


Using the frozen raw snapshot; no network calls were made.


## KSU 2025 source repair

The original 2025 KSU JSON contains one invalid `backslash + literal tab` sequence. The raw file is never edited; the pipeline creates a repaired copy in `data/interim/` when needed.

In [4]:
ksu_cfg = config["sources"]["ksu"]
source = paths.raw / ksu_cfg["files"][2025]
output = paths.interim / ksu_cfg["repaired_files"][2025]
repaired_path, repair_count = extract.repair_ksu_json(source, output)
print("Repair count:", repair_count)
print("Repaired copy:", repaired_path.relative_to(ROOT))


Repair count: 1
Repaired copy: data\interim\ksu_publications_2025_repaired.json
